
# **SQL Project 01 – AHCA Polling Analysis**
## **Dataset:** ahca_polls.csv (imported into SQLite as table ahca_polls)
## **Author:** Stephen Pryce

### Description:
   Exploratory analysis of polling data on the AHCA, including:
   - Pollster activity and performance
   - Overall favor vs oppose levels
   - Trends over time
   - Question text patterns
   - Data quality checks (missing URLs, odd totals)


In [2]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("polling_analysis_project.db")

SECTION 1: DATASET OVERVIEW
Basic counts and sample rows to understand the dataset size
and structure.

In [3]:
# Q01: Total number of polls in the dataset
# Purpose: Get the overall size of the dataset.

df = pd.read_sql("""
SELECT COUNT(*) AS total_polls
FROM ahca_polls
""", conn)

df

,total_polls
0,15


In [4]:
# Q02: Number of polls conducted by YouGov
# Purpose: See how many polls are attributed specifically to YouGov.

df = pd.read_sql("""
SELECT COUNT(*) AS yougov_polls
FROM ahca_polls
WHERE Pollster = 'YouGov'
""", conn)

df

,yougov_polls
0,4


In [5]:
# Q03: Sample of rows from the polling table
# Purpose: Inspect a few rows to understand columns and values.

df = pd.read_sql("""
SELECT *
FROM ahca_polls
LIMIT 10
""", conn)

df

,Start,End,Pollster,Favor,Oppose,Url,Text
0,6/9/17,6/11/17,Public Policy Polling,24,55,http://www.publicpolicypolling.com/pdf/2017/PP...,Do you support or oppose the health care bill ...
1,6/4/17,6/6/17,YouGov,33,45,https://d25d2506sfb94s.cloudfront.net/cumulus_...,Do you approve or disapprove of the American H...
2,5/31/17,6/6/17,Quinnipiac,17,62,https://poll.qu.edu/national/release-detail?Re...,There is a revised Republican health care plan...
3,5/26/17,5/30/17,IPSOS,30,46,https://www.realclearpolitics.com/docs/Core_Po...,Based on everything you know about the new Ame...
4,5/25/17,5/30/17,Morning Consult,38,47,https://morningconsult.com/wp-content/uploads/...,"As you may know, the House of Representatives ..."
5,5/17/17,5/23/17,Quinnipiac,20,57,https://poll.qu.edu/national/release-detail?Re...,There is a revised Republican health care plan...
6,5/16/17,5/22/17,Kaiser Family Foundation,31,55,http://files.kff.org/attachment/Topline-Kaiser...,"As you may know, Congress is currently discuss..."
7,5/13/17,5/20/17,Monmouth,32,55,https://www.monmouth.edu/polling-institute/rep...,"Earlier this month, the House of Representativ..."
8,5/13/17,5/16/17,YouGov,31,47,https://d25d2506sfb94s.cloudfront.net/cumulus_...,"Overall, given what you know about them, do yo..."
9,5/12/17,5/16/17,Public Policy Polling,25,52,http://www.publicpolicypolling.com/pdf/2017/PP...,Do you support or oppose the health care bill ...


SECTION 2: POLLSTERS & THEIR ACTIVITY
How many pollsters there are, who they are, and how busy they are.

In [6]:
# Q04: Number of distinct pollsters
# Purpose: Count how many unique pollsters appear in the dataset.

df = pd.read_sql("""
SELECT COUNT(DISTINCT Pollster) AS distinct_pollsters
FROM ahca_polls
""", conn)

df

,distinct_pollsters
0,8


In [7]:
# Q05: List of distinct pollster names
# Purpose: See the full list of pollsters in alphabetical order.

df = pd.read_sql("""
SELECT DISTINCT Pollster
FROM ahca_polls
ORDER BY Pollster
""", conn)

df

,Pollster
0,IPSOS
1,Kaiser Family Foundation
2,Monmouth
3,Morning Consult
4,NBC/WSJ
5,Public Policy Polling
6,Quinnipiac
7,YouGov


In [8]:
# Q06: Distinct pollster–favor combinations
# Purpose: Show unique combinations of Pollster and Favor value.
# Note: This can reveal the spread of Favor values per pollster.

df = pd.read_sql("""
SELECT DISTINCT
    Pollster,
    Favor
FROM ahca_polls
ORDER BY Favor
""", conn)

df

,Pollster,Favor
0,Quinnipiac,17
1,Quinnipiac,20
2,Quinnipiac,21
3,NBC/WSJ,23
4,Public Policy Polling,24
5,Public Policy Polling,25
6,IPSOS,30
7,Kaiser Family Foundation,31
8,YouGov,31
9,Monmouth,32


In [9]:
# Q07: Number of polls per pollster
# Purpose: See which pollsters are most active (by count of polls).

df = pd.read_sql("""
SELECT
    Pollster,
    COUNT(*) AS num_polls
FROM ahca_polls
GROUP BY Pollster
ORDER BY num_polls DESC
""", conn)

df

,Pollster,num_polls
0,YouGov,4
1,Quinnipiac,3
2,Public Policy Polling,2
3,Morning Consult,2
4,NBC/WSJ,1
5,Monmouth,1
6,Kaiser Family Foundation,1
7,IPSOS,1


In [10]:
# Q08: Average Favor per pollster
# Purpose: Compare pollsters based on average Favor scores.

df = pd.read_sql("""
SELECT
    Pollster,
    AVG(Favor) AS avg_favor
FROM ahca_polls
GROUP BY Pollster
ORDER BY avg_favor DESC
""", conn)

df

,Pollster,avg_favor
0,Morning Consult,38.000000
1,Monmouth,32.000000
2,YouGov,31.500000
3,Kaiser Family Foundation,31.000000
4,IPSOS,30.000000
5,Public Policy Polling,24.500000
6,NBC/WSJ,23.000000
7,Quinnipiac,19.333333


In [11]:
# Q09: Total Favor per pollster
# Purpose: Sum of Favor values per pollster (workload × support).

df = pd.read_sql("""
SELECT
    Pollster,
    SUM(Favor) AS total_favor
FROM ahca_polls
GROUP BY Pollster
ORDER BY total_favor DESC
""", conn)

df

,Pollster,total_favor
0,YouGov,126
1,Morning Consult,76
2,Quinnipiac,58
3,Public Policy Polling,49
4,Monmouth,32
5,Kaiser Family Foundation,31
6,IPSOS,30
7,NBC/WSJ,23


In [12]:
# Q10: Average Favor and Oppose per pollster
# Purpose: Compare pollsters on both support and opposition.

df = pd.read_sql("""
SELECT
    Pollster,
    AVG(Favor) AS avg_favor,
    AVG(Oppose) AS avg_oppose
FROM ahca_polls
GROUP BY Pollster
ORDER BY avg_favor DESC
""", conn)

df

,Pollster,avg_favor,avg_oppose
0,Morning Consult,38.000000,45.500000
1,Monmouth,32.000000,55.000000
2,YouGov,31.500000,45.750000
3,Kaiser Family Foundation,31.000000,55.000000
4,IPSOS,30.000000,46.000000
5,Public Policy Polling,24.500000,53.500000
6,NBC/WSJ,23.000000,48.000000
7,Quinnipiac,19.333333,58.333333


In [13]:
# Q11: Descriptive statistics per pollster
# Purpose: For each pollster, show:
# - number of polls
# - average favor
# - average oppose
# - first poll date
# - last poll date

df = pd.read_sql("""
SELECT
    Pollster,
    COUNT(*) AS num_polls,
    AVG(Favor) AS avg_favor,
    AVG(Oppose) AS avg_oppose,
    MIN(Start) AS first_poll,
    MAX(End) AS last_poll
FROM ahca_polls
GROUP BY Pollster
ORDER BY num_polls DESC
""", conn)

df

,Pollster,num_polls,avg_favor,avg_oppose,first_poll,last_poll
0,YouGov,4,31.500000,45.750000,5/13/17,6/6/17
1,Quinnipiac,3,19.333333,58.333333,5/17/17,6/6/17
2,Public Policy Polling,2,24.500000,53.500000,5/12/17,6/11/17
3,Morning Consult,2,38.000000,45.500000,5/25/17,5/9/17
4,NBC/WSJ,1,23.000000,48.000000,5/11/17,5/15/17
5,Monmouth,1,32.000000,55.000000,5/13/17,5/20/17
6,Kaiser Family Foundation,1,31.000000,55.000000,5/16/17,5/22/17
7,IPSOS,1,30.000000,46.000000,5/26/17,5/30/17


In [14]:
# Q12: All rows for a specific pollster (IPSOS)
# Purpose: Drill down into one pollster’s individual polls.

df = pd.read_sql("""
SELECT *
FROM ahca_polls
WHERE Pollster = 'IPSOS'
""", conn)

df

,Start,End,Pollster,Favor,Oppose,Url,Text
0,5/26/17,5/30/17,IPSOS,30,46,https://www.realclearpolitics.com/docs/Core_Po...,Based on everything you know about the new Ame...


SECTION 3: OVERALL OPINION LEVELS
Aggregate favor/oppose values across the entire dataset.

In [15]:
# Q13: Average Favor and Oppose (overall), rounded to 1 decimal
# Purpose: Get central tendency of support vs opposition.
df = pd.read_sql("""
SELECT
    ROUND(AVG(Favor),  1) AS avg_favor,
    ROUND(AVG(Oppose), 1) AS avg_oppose
FROM ahca_polls;
""", conn)

df

,avg_favor,avg_oppose
0,28.3,50.7


In [16]:
# Q14: Total Favor and total Oppose (overall)
# Purpose: Sum all Favor and Oppose values across the dataset.
df = pd.read_sql("""
SELECT
    SUM(Favor)  AS total_favor,
    SUM(Oppose) AS total_oppose
FROM ahca_polls;
""", conn)

df

,total_favor,total_oppose
0,425,760


SECTION 4: DATE RANGE & TRENDS OVER TIME
Poll start/end dates and monthly favor/oppose trends.

In [17]:
# Q15: Most recent poll end date
# Purpose: See how up-to-date the polling data is.
df = pd.read_sql("""
SELECT 
    MAX(End) AS most_recent_poll_end
FROM ahca_polls;
""", conn)

df

,most_recent_poll_end
0,6/6/17


In [18]:
# Q16: Earliest poll start date
# Purpose: See when the polling period begins.
df = pd.read_sql("""
SELECT 
    MIN(Start) AS earliest_poll_start
FROM ahca_polls;
""", conn)

df

,earliest_poll_start
0,5/11/17


In [19]:
# Q17: Monthly average Favor and Oppose
# Purpose: Create a month value from Start date (assumes MM/DD/…)
#          and compute average Favor and Oppose per month.
df = pd.read_sql("""
SELECT
    SUBSTR(Start, 1, 2)            AS month,
    ROUND(AVG(Favor),  1)          AS avg_favor,
    ROUND(AVG(Oppose), 1)          AS avg_oppose
FROM ahca_polls
GROUP BY month
ORDER BY month;
""", conn)

df

,month,avg_favor,avg_oppose
0,5/,28.3,50.8
1,6/,28.5,50.0


SECTION 5: QUESTION TEXT ANALYSIS
Look at the distinct question wordings and health-care-related text.

In [20]:
# Q18: Count of distinct question texts
# Purpose: See how many unique question wordings are used.
df = pd.read_sql("""
SELECT 
    COUNT(DISTINCT Text) AS distinct_text_count
FROM ahca_polls;
""", conn)

df

,distinct_text_count
0,11


In [21]:
# Q19: List of distinct question texts
# Purpose: Inspect all unique survey question texts.
df = pd.read_sql("""
SELECT DISTINCT
    Text
FROM ahca_polls
ORDER BY Text;
""", conn)

df

,Text
0,"As you may know, Congress is currently discuss..."
1,"As you may know, Republican leaders in the Hou..."
2,"As you may know, the House of Representatives ..."
3,Based on everything you know about the new Ame...
4,Do you approve or disapprove of the American H...
5,Do you support or oppose the health care bill ...
6,Do you support or oppose the health care bill ...
7,"Earlier this month, the House of Representativ..."
8,"Now as you may know, health care legislation w..."
9,"Overall, given what you know about them, do yo..."


In [22]:
# Q20: Polls where the question text mentions "health care"
# Purpose: Filter polls whose text references health care explicitly.
df = pd.read_sql("""
SELECT 
    *
FROM ahca_polls
WHERE Text LIKE '%health care%';
""", conn)

df

,Start,End,Pollster,Favor,Oppose,Url,Text
0,6/9/17,6/11/17,Public Policy Polling,24,55,http://www.publicpolicypolling.com/pdf/2017/PP...,Do you support or oppose the health care bill ...
1,6/4/17,6/6/17,YouGov,33,45,https://d25d2506sfb94s.cloudfront.net/cumulus_...,Do you approve or disapprove of the American H...
2,5/31/17,6/6/17,Quinnipiac,17,62,https://poll.qu.edu/national/release-detail?Re...,There is a revised Republican health care plan...
3,5/26/17,5/30/17,IPSOS,30,46,https://www.realclearpolitics.com/docs/Core_Po...,Based on everything you know about the new Ame...
4,5/25/17,5/30/17,Morning Consult,38,47,https://morningconsult.com/wp-content/uploads/...,"As you may know, the House of Representatives ..."
5,5/17/17,5/23/17,Quinnipiac,20,57,https://poll.qu.edu/national/release-detail?Re...,There is a revised Republican health care plan...
6,5/16/17,5/22/17,Kaiser Family Foundation,31,55,http://files.kff.org/attachment/Topline-Kaiser...,"As you may know, Congress is currently discuss..."
7,5/13/17,5/20/17,Monmouth,32,55,https://www.monmouth.edu/polling-institute/rep...,"Earlier this month, the House of Representativ..."
8,5/13/17,5/16/17,YouGov,31,47,https://d25d2506sfb94s.cloudfront.net/cumulus_...,"Overall, given what you know about them, do yo..."
9,5/12/17,5/16/17,Public Policy Polling,25,52,http://www.publicpolicypolling.com/pdf/2017/PP...,Do you support or oppose the health care bill ...


SECTION 6: FAVOR VS OPPOSE OUTCOMES
Compare polls where Favor > Oppose vs Oppose > Favor, and extremes.

In [23]:
# Q21: Polls where Favor is greater than Oppose
# Purpose: Identify polls with net positive support.
df = pd.read_sql("""
SELECT 
    *
FROM ahca_polls
WHERE Favor > Oppose;
""", conn)

df

,Start,End,Pollster,Favor,Oppose,Url,Text


In [24]:
# Q22: Polls where Oppose is greater than Favor
# Purpose: Identify polls with net opposition.
df = pd.read_sql("""
SELECT 
    *
FROM ahca_polls
WHERE Oppose > Favor;
""", conn)

df

,Start,End,Pollster,Favor,Oppose,Url,Text
0,6/9/17,6/11/17,Public Policy Polling,24,55,http://www.publicpolicypolling.com/pdf/2017/PP...,Do you support or oppose the health care bill ...
1,6/4/17,6/6/17,YouGov,33,45,https://d25d2506sfb94s.cloudfront.net/cumulus_...,Do you approve or disapprove of the American H...
2,5/31/17,6/6/17,Quinnipiac,17,62,https://poll.qu.edu/national/release-detail?Re...,There is a revised Republican health care plan...
3,5/26/17,5/30/17,IPSOS,30,46,https://www.realclearpolitics.com/docs/Core_Po...,Based on everything you know about the new Ame...
4,5/25/17,5/30/17,Morning Consult,38,47,https://morningconsult.com/wp-content/uploads/...,"As you may know, the House of Representatives ..."
5,5/17/17,5/23/17,Quinnipiac,20,57,https://poll.qu.edu/national/release-detail?Re...,There is a revised Republican health care plan...
6,5/16/17,5/22/17,Kaiser Family Foundation,31,55,http://files.kff.org/attachment/Topline-Kaiser...,"As you may know, Congress is currently discuss..."
7,5/13/17,5/20/17,Monmouth,32,55,https://www.monmouth.edu/polling-institute/rep...,"Earlier this month, the House of Representativ..."
8,5/13/17,5/16/17,YouGov,31,47,https://d25d2506sfb94s.cloudfront.net/cumulus_...,"Overall, given what you know about them, do yo..."
9,5/12/17,5/16/17,Public Policy Polling,25,52,http://www.publicpolicypolling.com/pdf/2017/PP...,Do you support or oppose the health care bill ...


In [25]:
# Q23: Count of polls where Oppose > Favor
# Purpose: How many polls show net opposition?
df = pd.read_sql("""
SELECT 
    COUNT(*) AS polls_oppose_greater
FROM ahca_polls
WHERE Oppose > Favor;
""", conn)

df

,polls_oppose_greater
0,15


In [26]:
# Q24: Poll with the highest Favor value
# Purpose: Find the single most favorable poll result.
df = pd.read_sql("""
SELECT 
    *
FROM ahca_polls
ORDER BY Favor DESC
LIMIT 1;
""", conn)

df

,Start,End,Pollster,Favor,Oppose,Url,Text
0,5/25/17,5/30/17,Morning Consult,38,47,https://morningconsult.com/wp-content/uploads/...,"As you may know, the House of Representatives ..."


In [27]:
# Q25: Poll with the highest Oppose value
# Purpose: Find the single strongest opposition result.
df = pd.read_sql("""
SELECT 
    *
FROM ahca_polls
ORDER BY Oppose DESC
LIMIT 1;
""", conn)

df

,Start,End,Pollster,Favor,Oppose,Url,Text
0,5/31/17,6/6/17,Quinnipiac,17,62,https://poll.qu.edu/national/release-detail?Re...,There is a revised Republican health care plan...


In [28]:
# Q26: Polls with unusual total (Favor + Oppose)
# Purpose: Flag polls where Favor + Oppose is < 75 or > 95,
#          which may indicate undecided/other response patterns
#          or potential data issues.
df = pd.read_sql("""
SELECT 
    *
FROM ahca_polls
WHERE (Favor + Oppose) < 75
   OR (Favor + Oppose) > 95;
   """, conn)

df

,Start,End,Pollster,Favor,Oppose,Url,Text
0,5/11/17,5/15/17,NBC/WSJ,23,48,https://www.scribd.com/document/348302118/NBC-...,"Now as you may know, health care legislation w..."


SECTION 7: URL COMPLETENESS & DATA QUALITY
Check which polls have missing or present URLs.

In [29]:
# Q27: Count of polls without a URL
# Purpose: Measure how many polls lack a web link (URL missing or blank).
df = pd.read_sql("""
SELECT 
    COUNT(*) AS polls_without_url
FROM ahca_polls
WHERE Url IS NULL OR Url = '';
""", conn)

df

,polls_without_url
0,0


In [30]:
# Q28: Pollster and URL where a URL is present
# Purpose: List pollsters with associated URLs (non-empty).
df = pd.read_sql("""
SELECT 
    Pollster,
    Url
FROM ahca_polls
WHERE Url IS NOT NULL
  AND Url <> '';
""", conn)

df

,Pollster,Url
0,Public Policy Polling,http://www.publicpolicypolling.com/pdf/2017/PP...
1,YouGov,https://d25d2506sfb94s.cloudfront.net/cumulus_...
2,Quinnipiac,https://poll.qu.edu/national/release-detail?Re...
3,IPSOS,https://www.realclearpolitics.com/docs/Core_Po...
4,Morning Consult,https://morningconsult.com/wp-content/uploads/...
5,Quinnipiac,https://poll.qu.edu/national/release-detail?Re...
6,Kaiser Family Foundation,http://files.kff.org/attachment/Topline-Kaiser...
7,Monmouth,https://www.monmouth.edu/polling-institute/rep...
8,YouGov,https://d25d2506sfb94s.cloudfront.net/cumulus_...
9,Public Policy Polling,http://www.publicpolicypolling.com/pdf/2017/PP...
